> **ЗАСТАРІЛО.** Цей Colab-ноутбук викликає старий API `extract_document` і більше не працює з поточним кодом. Канонічний шлях -- локальний запуск `python run_pipeline.py` (див. `README.md` і `context/extraction-pipeline-prototype.md`). Залишений як історія експерименту.

# Пробний прогін пайплайна -- generic-двигун (v6)

Google Colab. Перед запуском: `Runtime > Change runtime type > GPU` (12B на GPU-рантаймі суттєво швидше за CPU -- код автоматично збирає `llama-cpp-python` з CUDA, якщо `nvcc` присутній; після зміни типу рантайму сесію треба перезапустити, інакше лишається стара CPU-збірка).

**Зміни проти v5 -- гібридна екстракція замість "по одному LLM-виклику на поле":**
1. **Спершу дешевий детермінований прохід** для всіх полів (як і раніше). Те, що він не закрив ("прогалини"), збирається й іде далі.
2. **Один grammar-constrained LLM-виклик на ГРУПУ полів** (за замовчуванням 4), а не на кожне поле окремо й не на всі одразу: компроміс між швидкістю (менше викликів) і ізоляцією (збій однієї групи не валить документ цілком). JSON Schema для grammar будується автоматично зі схеми + довідників (`pipeline/extraction/schema_grammar.py`) -- категоріальні поля обмежені реальними кодами, LLM фізично не може вигадати неіснуючий.
3. **Прізвище не у ВЕЛИКОМУ регістрі** більше не ламає given_name/patronymic зсувом -- якщо регістр не дав однозначної відповіді, усі три поля йдуть у LLM-групу разом (`pipeline/extraction/extract.py:parse_rank_and_name`).
4. **Класифікація домену з LLM-фолбеком тепер спрацьовує автоматично** (не вручну), і LLM явно може відповісти "unknown" (grammar включає цю опцію) -- раніше grammar-обмежений вибір змусив би модель вибрати щось із закритого списку навіть коли жоден варіант не підходить.
5. **Перевірка узгодженості схема↔домен** перед фінальною екстракцією -- явний WARNING, якщо завантажена схема не того домену, що визначила класифікація.
6. **Перевірка довідників** -- явний WARNING, якщо схема посилається на `category`, довідник якої не завантажено.
7. **`subject`** (був `person_fields`) + **`document_meta`** тепер повна YAML-шапка з усіма витягнутими полями (не окремий JSON-блок у тілі `.md`).
8. **docx: header/footer тепер читаються** (кутовий штамп реального документа міг бути там, а не в тілі).
9. Розширене розпізнавання "порожньо"-маркерів бланка (голі підкреслення/риски, "не заповнено" тощо).

## 0. Залежності

In [ ]:
!pip install -q surya-ocr pyyaml

### 0.1. Збірка `llama-server` (потрібно для роботи самого Surya)

In [ ]:
import os, glob, subprocess, shutil

if not os.path.isdir("/content/llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)

has_cuda = shutil.which("nvcc") is not None
print("nvcc знайдено:", has_cuda)

cmake_args = ["cmake", "-B", "/content/llama.cpp/build", "-S", "/content/llama.cpp",
              "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"]
cmake_args.append("-DGGML_CUDA=ON" if has_cuda else "-DGGML_CUDA=OFF")

result = subprocess.run(cmake_args, capture_output=True, text=True)
print(result.stdout[-3000:]); print(result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError("cmake configure впав -- див. лог вище")

build = subprocess.run(["cmake", "--build", "/content/llama.cpp/build", "--config", "Release",
                         "-j", str(os.cpu_count()), "--target", "llama-server"],
                        capture_output=True, text=True)
print(build.stdout[-3000:]); print(build.stderr[-3000:])
if build.returncode != 0:
    raise RuntimeError("cmake build впав -- див. лог вище")

candidates = glob.glob("/content/llama.cpp/build/**/llama-server", recursive=True)
assert candidates, "llama-server не знайдено після збірки"
os.environ["LLAMA_CPP_BINARY"] = candidates[0]
print("llama-server:", candidates[0])

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import list_repo_files, hf_hub_download
import os

REPO_ID = "INSAIT-Institute/MamayLM-Gemma-3-12B-IT-v2.0-GGUF"  # звірте на huggingface.co перед запуском
FILENAME = "MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M.gguf"
DEST_DIR = "/content/drive/MyDrive/models"

os.makedirs(DEST_DIR, exist_ok=True)

available = list_repo_files(REPO_ID)
if FILENAME not in available:
    raise FileNotFoundError(
        f"'{FILENAME}' немає в репозиторії {REPO_ID}. Доступні файли:\n" + "\n".join(available)
    )

model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir=DEST_DIR)
print("Завантажено:", model_path)
print("Розмір, МБ:", round(os.path.getsize(model_path) / 1024**2, 1))

## 1. Завантаження файлів

Потрібно завантажити одразу все:
1. Фото/скан документа (будь-якого домену)
2. `schemas/<template>.yaml` — схема саме цього шаблону
3. Усі `dictionaries/*.yaml`, на які посилається схема (`category:` у полях) + `dictionaries/domain_keyphrases.yaml`
4. Файли двигуна: `pipeline/classification/classify.py`, `pipeline/extraction/extract.py`, `pipeline/extraction/schema_grammar.py`, `pipeline/normalization/normalize.py`, `pipeline/build_record.py`, `pipeline/ingestion/ingest.py`
5. (необов'язково) `pipeline/llm_context/document_processing_guidelines.md`

Після завантаження файли двигуна розкладаються у структуру пакета, щоб `import pipeline...` працював без змін відносно самого проєкту.

In [ ]:
from google.colab import files
import os, shutil, sys, glob

os.makedirs("/content/input", exist_ok=True)
print("Завантажте: фото + schema.yaml + усі потрібні dictionaries/*.yaml + "
      "classify.py + extract.py + schema_grammar.py + normalize.py + build_record.py + "
      "ingest.py + (опційно) document_processing_guidelines.md")
uploaded = files.upload()
for name in uploaded:
    os.replace(name, f"/content/input/{name}")

# --- розкладаємо файли двигуна у пакетну структуру ---
PKG_ROOT = "/content"
for pkg in ["pipeline", "pipeline/classification", "pipeline/extraction",
            "pipeline/normalization", "pipeline/ingestion"]:
    os.makedirs(f"{PKG_ROOT}/{pkg}", exist_ok=True)
    open(f"{PKG_ROOT}/{pkg}/__init__.py", "a").close()

PY_TARGETS = {
    "classify.py": "pipeline/classification/classify.py",
    "extract.py": "pipeline/extraction/extract.py",
    "schema_grammar.py": "pipeline/extraction/schema_grammar.py",
    "normalize.py": "pipeline/normalization/normalize.py",
    "build_record.py": "pipeline/build_record.py",
    "ingest.py": "pipeline/ingestion/ingest.py",
}
for src_name, rel_target in PY_TARGETS.items():
    src = f"/content/input/{src_name}"
    if os.path.exists(src):
        shutil.copy(src, f"{PKG_ROOT}/{rel_target}")

if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)

# IMAGE_PATH: скануємо /content/input НАПРЯМУ, а не `uploaded` -- `uploaded`
# це лише те, що вибрано в ОСТАННЬОМУ виклику files.upload(); якщо цю
# клітинку перезапускали без повторного вибору фото, воно фізично лишається
# на диску, але `uploaded` про нього вже не знає.
IMAGE_PATH = next((p for p in glob.glob("/content/input/*")
                    if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))), None)
print("IMAGE_PATH:", IMAGE_PATH)

# SCHEMA_PATH визначається за ВМІСТОМ, не назвою файлу -- див. клітинку 4:
# назва довідника (напр. "military_rank.yaml") не містить жодного
# розпізнаваного маркера "це не схема", а порядок glob.glob() не
# гарантовано алфавітний, тож пошук "перший .yaml, що не схожий на
# довідник за назвою" міг випадково взяти довідник замість схеми.

## 2. OCR (Surya)

In [ ]:
import os, hashlib
os.environ.setdefault("SURYA_INFERENCE_PARALLEL", "2")

import re, html, glob
from PIL import Image, ImageOps
from surya.inference import SuryaInferenceManager
from surya.recognition import RecognitionPredictor
from pipeline.ingestion.ingest import sort_blocks_by_geometry

manager = SuryaInferenceManager()
recognition_predictor = RecognitionPredictor(manager)

def load_image_for_ocr(image_path: str) -> Image.Image:
    return ImageOps.exif_transpose(Image.open(image_path)).convert("RGB")

def file_sha256(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

def run_surya(image_path: str):
    """Зберігає bbox кожного блоку. getattr(..., None), а не block.bbox
    напряму -- якщо інша версія/режим Surya колись перестане повертати bbox,
    краще явна ValueError у sort_blocks_by_geometry (ingest.py), ніж
    AttributeError тут чи тиха робота без геометрії."""
    predictions = recognition_predictor([load_image_for_ocr(image_path)])
    raw_blocks = []
    for block in predictions[0].blocks:
        plain = re.sub(r"<br\s*/?>", "\n", block.html)
        plain = re.sub(r"<[^>]+>", "", plain)
        plain = html.unescape(plain).strip()
        if plain:
            raw_blocks.append({"text": plain, "bbox": getattr(block, "bbox", None)})
    ocr_blocks = sort_blocks_by_geometry(raw_blocks)
    return "\n".join(ocr_blocks), ocr_blocks

if not IMAGE_PATH:
    candidates = [p for p in glob.glob("/content/input/*")
                  if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"))]
    assert candidates, "Не знайдено зображення в /content/input"
    IMAGE_PATH = candidates[0]
    print("IMAGE_PATH автовизначено:", IMAGE_PATH)

FILE_HASH = file_sha256(IMAGE_PATH)
ocr_text, ocr_blocks = run_surya(IMAGE_PATH)
print("file_hash:", FILE_HASH)
for i, b in enumerate(ocr_blocks):
    print(f"[{i}]", repr(b))

## 3. Класифікація домену (A: заголовок, B: фрази тіла, C: LLM третьою)

Генерично: усі фрази читаються з `dictionaries/domain_keyphrases.yaml`, код нічого не знає про конкретні домени.

In [ ]:
import yaml
from pipeline.classification.classify import load_domain_keyphrases, classify_domain

DOMAINS = load_domain_keyphrases("/content/input/domain_keyphrases.yaml")
domain, scores, source = classify_domain(ocr_text, DOMAINS)  # llm_classify додається автоматично в кроці 6, якщо тут None
print("Скори:", scores, "| Домен:", domain, "| Джерело:", source)
if domain is None:
    print("A+B не дали збігу -- крок 6 (LLM) спробує ще раз автоматично, вручну нічого перезапускати не треба")

## 4. Завантаження схеми й довідників

In [ ]:
from pipeline.normalization.normalize import build_alias_lookup
import glob, yaml

# Класифікуємо кожен завантажений .yaml за ВМІСТОМ, не назвою файлу:
# схема -- має "template" і "fields"; довідник значень -- "category" і
# "values"; domain_keyphrases.yaml -- "domains" (уже завантажений окремо
# на кроці 3, тут просто пропускаємо за назвою, бо форма унікальна й
# сталого імені).
SCHEMA_PATH = None
DICTIONARY_PATHS = []
for path in glob.glob("/content/input/*.yaml"):
    if os.path.basename(path) == "domain_keyphrases.yaml":
        continue
    with open(path, encoding="utf-8") as f:
        content = yaml.safe_load(f)
    if not isinstance(content, dict):
        continue
    if "template" in content and "fields" in content:
        if SCHEMA_PATH is not None:
            raise ValueError(f"Знайдено кілька файлів-схем: {SCHEMA_PATH} і {path} -- завантажте лише одну схему за раз")
        SCHEMA_PATH = path
    elif "category" in content and "values" in content:
        DICTIONARY_PATHS.append(path)

assert SCHEMA_PATH is not None, "Не знайдено файл схеми (потрібні ключі 'template' і 'fields') серед завантажених .yaml"
print("SCHEMA_PATH:", SCHEMA_PATH)

with open(SCHEMA_PATH, encoding="utf-8") as f:
    schema = yaml.safe_load(f)

# Схема↔домен: якщо класифікація (крок 3) вже визначила домен і він НЕ
# збігається з domain у схемі -- людина, найімовірніше, підсунула не той
# файл. Раніше ніщо це не перевіряло: документ проходив увесь пайплайн
# із чужою схемою мовчки, повертаючи купу no_value замість явного сигналу.
if domain is not None and schema.get("domain") != domain:
    print(f"УВАГА: класифікація визначила домен '{domain}', а схема '{SCHEMA_PATH}' "
          f"належить домену '{schema.get('domain')}' -- ймовірно, завантажена НЕ та схема.")

DICTIONARIES = {}
for path in DICTIONARY_PATHS:
    with open(path, encoding="utf-8") as f:
        raw_dict = yaml.safe_load(f)
    DICTIONARIES[raw_dict["category"]] = build_alias_lookup(raw_dict)

print("Схема:", schema["template"], "| Домен схеми:", schema["domain"], "| fact_type:", schema.get("fact_type"))
print("Поля:", [f["name"] for f in schema["fields"]])
print("Завантажені довідники:", list(DICTIONARIES))

# Довідники: усі "category"-поля схеми мають мати відповідний завантажений
# довідник -- інакше поле мовчки лишиться "unknown" без жодного пояснення,
# чому саме. Явний WARNING замість тихого {}.
required_categories = {f["category"] for f in schema["fields"] if f.get("type") == "category"}
missing_categories = required_categories - set(DICTIONARIES)
if missing_categories:
    print("УВАГА: для схеми не завантажено довідники категорій:", missing_categories)

_rank_field = next((f for f in schema["fields"] if f["name"] == "rank"), None)
rank_alias_lookup = DICTIONARIES.get(_rank_field["category"], {}) if _rank_field else {}


## 5. Екстракція (генерично, за схемою)

In [ ]:
from pipeline.extraction.extract import extract_document

TITLE_PHRASES = DOMAINS.get(domain, {}).get("title", []) if domain else []

raw_extraction = extract_document(schema, ocr_text, ocr_blocks, DICTIONARIES,
                                   title_phrases=TITLE_PHRASES)
for k, v in raw_extraction.items():
    print(k, "->", v)
gap_fields = [k for k, (v, reason) in raw_extraction.items() if v is None and reason != "derived"]
print("\nПоля-\"прогалини\" (потребують LLM, крок 6):", gap_fields)

## 6. LLM (MamayLM) — safety net і для класифікації, і для `extraction: llm` полів

In [ ]:
import os, shutil, subprocess

has_cuda = shutil.which("nvcc") is not None
print("nvcc знайдено:", has_cuda)
env = os.environ.copy()
if has_cuda:
    env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
else:
    print("nvcc відсутній -- ставимо llama-cpp-python без примусової CUDA-збірки (CPU)")
subprocess.run(["pip", "install", "-q", "llama-cpp-python", "--no-cache-dir",
                "--force-reinstall", "--upgrade"], env=env, check=True)

In [ ]:
import json as _json
from pipeline.classification.classify import classify_domain

MODEL_PATH = "/content/drive/MyDrive/models/MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M.gguf"  # скоригуйте шлях
LLM_CONTEXT_PATH = "/content/input/document_processing_guidelines.md"
SELF_CONSISTENCY_N = 1  # >1 вмикає majority_vote по кожній групі; grammar вже прибирає найгрубіший клас помилок (невалідна структура/категорія), тому 1 -- прийнятний старт
BATCH_SIZE = 4  # полів на один LLM-виклик -- компроміс швидкість/ізоляція збою, див. markdown на початку ноутбука

def load_llm_context(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"УВАГА: {path} не знайдено -- LLM працюватиме без системного промпту")
        return ""

LLM_SYSTEM_PROMPT = load_llm_context(LLM_CONTEXT_PATH)
_llm = None

def get_llm():
    global _llm
    if _llm is None:
        from llama_cpp import Llama
        _llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=-1, chat_format="gemma")
    return _llm

def _grammar_from_choices(choices):
    from llama_cpp import LlamaGrammar
    escaped = [c.replace("\\", "\\\\").replace('"', '\\"') for c in choices]
    rule = " | ".join(f'"{c}"' for c in escaped)
    return LlamaGrammar.from_string(f"root ::= {rule}")

def llm_classify(prompt: str, choices: list) -> str:
    """Grammar-constrained вибір РІВНО одного з choices -- модель фізично
    не може відповісти нічим іншим (включно з "unknown", якщо він у
    choices). Викликається з pipeline.classification.classify_domain_llm."""
    llm = get_llm()
    messages = []
    if LLM_SYSTEM_PROMPT:
        messages.append({"role": "system", "content": LLM_SYSTEM_PROMPT})
    messages.append({"role": "user", "content": prompt})
    resp = llm.create_chat_completion(messages=messages, max_tokens=16, temperature=0,
                                       grammar=_grammar_from_choices(choices))
    return resp["choices"][0]["message"]["content"].strip()

def llm_extract_batch(field_defs: list, ocr_text: str, json_schema: dict) -> dict:
    """Один LLM-виклик на ГРУПУ полів (field_defs), grammar-constrained за
    json_schema (побудована в pipeline.extraction.schema_grammar). Кожне
    поле в json_schema nullable -- модель може чесно повернути null замість
    вигадування значення, якого в тексті немає."""
    from llama_cpp import LlamaGrammar
    llm = get_llm()
    grammar = LlamaGrammar.from_json_schema(_json.dumps(json_schema))
    field_descriptions = "\n".join(
        f"- {f['name']}" + (f" ({f['note']})" if f.get("note") else "")
        for f in field_defs
    )
    messages = []
    if LLM_SYSTEM_PROMPT:
        messages.append({"role": "system", "content": LLM_SYSTEM_PROMPT})
    messages.append({
        "role": "user",
        "content": (
            "З наведеного тексту документа витягни значення полів нижче. "
            "Якщо значення поля в тексті немає -- поверни null для цього поля, "
            "не вигадуй. Відповідай лише JSON-об'єктом.\n\n"
            f"Поля:\n{field_descriptions}\n\nТекст документа:\n{ocr_text}"
        ),
    })
    resp = llm.create_chat_completion(messages=messages, max_tokens=64 * max(1, len(field_defs)),
                                       temperature=0.0 if SELF_CONSISTENCY_N <= 1 else 0.7,
                                       grammar=grammar)
    return _json.loads(resp["choices"][0]["message"]["content"])

print("LLM-функції готові (grammar-constrained класифікація + групова екстракція, "
      "BATCH_SIZE =", BATCH_SIZE, ", SELF_CONSISTENCY_N =", SELF_CONSISTENCY_N, "). "
      "Системний промпт:", "завантажено" if LLM_SYSTEM_PROMPT else "відсутній")

# Якщо крок 3 не визначив домен -- пробуємо ще раз автоматично, тепер з LLM.
# Раніше це вимагало вручну відредагувати й перезапустити клітинку 3.
if domain is None:
    domain, scores, source = classify_domain(ocr_text, DOMAINS, llm_classify=llm_classify)
    print("Повторна класифікація (LLM):", "Домен:", domain, "| Джерело:", source)
    TITLE_PHRASES = DOMAINS.get(domain, {}).get("title", []) if domain else []
    if domain is not None and schema.get("domain") != domain:
        print(f"УВАГА: LLM визначила домен '{domain}', а завантажена схема -- домену "
              f"'{schema.get('domain')}'. Перевірте, чи це справді той самий документ.")

if domain is None:
    print("\nДомен не визначено навіть із LLM -- документ іде як UNRESOLVED "
          "(крок 7 збереже OCR-текст + хеш у чергу, без спроби екстракції).")

# Повторюємо екстракцію -- "прогалини" з кроку 5 тепер підуть у LLM-групи.
raw_extraction = extract_document(schema, ocr_text, ocr_blocks, DICTIONARIES,
                                   llm_extract_batch=llm_extract_batch,
                                   title_phrases=TITLE_PHRASES,
                                   batch_size=BATCH_SIZE,
                                   self_consistency_n=SELF_CONSISTENCY_N)
print("\nОновлена екстракція:")
for k, v in raw_extraction.items():
    print(k, "->", v)

## 7. Підсумковий запис — розкладений за структурою БД (генерично, за `db_target` зі схеми)

In [ ]:
import uuid, datetime, json
from pipeline.build_record import build_record

DOCUMENT_ID = str(uuid.uuid4())
UPLOADED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat()

if domain is None:
    # Документ не вдалося класифікувати навіть із LLM -- не намагаємось
    # витягувати поля з (можливо) невідповідної схеми. Явна UNRESOLVED-
    # гілка замість тихого запису зі сміттєвими полями чи краху.
    UNRESOLVED = True
    document_meta = {
        "id": DOCUMENT_ID,
        "status": "unresolved",
        "file_hash": FILE_HASH,
        "uploaded_at": UPLOADED_AT,
        "reason": "domain not classified (rules + LLM)",
    }
    print("=== UNRESOLVED -- документ у чергу на розгляд ===")
    print(json.dumps(document_meta, ensure_ascii=False, indent=2))
else:
    UNRESOLVED = False
    subject, fact, unknown_fields, confirmed_empty_fields, not_implemented_fields = build_record(
        schema, raw_extraction, DICTIONARIES
    )
    fact["source_document_id"] = DOCUMENT_ID

    # document_meta -- ПОВНА шапка: ідентифікація документа + subject + fact
    # разом, одним об'єктом (раніше subject/fact йшли окремим JSON-блоком у
    # тілі .md, а document_meta був лише тонкою ідентифікацією).
    document_meta = {
        "id": DOCUMENT_ID,
        "status": "confirmed" if fact["confirmed"] else "needs_review",
        "domain": domain,
        "template": schema.get("template"),
        "file_hash": FILE_HASH,
        "uploaded_at": UPLOADED_AT,
        "subject": subject,
        "fact": fact,
        "unknown_fields": unknown_fields,
        "confirmed_empty_fields": confirmed_empty_fields,
        "not_implemented_fields": not_implemented_fields,
    }

    print(json.dumps(document_meta, ensure_ascii=False, indent=2))

## 7b. Збірка `.md`-файлу для MinIO

In [ ]:
# document_meta вже містить УСІ витягнуті поля -- .md більше не має
# окремого JSON-блоку в тілі (раніше дублювало те, що мало бути в шапці).
minio_dir = "unresolved" if UNRESOLVED else document_meta["domain"]
minio_path = f"documents/{minio_dir}/{document_meta['id']}.md"

minio_md_content = (
    "---\n"
    + yaml.safe_dump(document_meta, allow_unicode=True, sort_keys=False)
    + "---\n\n"
    + "## OCR-текст\n\n"
    + ocr_text
    + "\n"
)

print("шлях у MinIO:", minio_path)
print()
print(minio_md_content)

## 8. Звірка на око — і що досі НЕ реалізовано

`unknown_fields` — поля, які мали значення спробувати, але не вийшло (включно з `llm_error:*`, якщо збій стався саме на LLM-групі). `confirmed_empty_fields` — підтверджено відсутнє. `not_implemented_fields` — деклароване в схемі (`priority: deferred`), без коду екстракції; не впливає на `fact.confirmed`. `document_meta["status"]` — `confirmed` / `needs_review` / `unresolved` (домен не визначено навіть із LLM).

**Свідомо не реалізовано незалежно від домену:**
- Резолюція `object_id`/`object_ref`-полів проти реального реєстру об'єктів — немає заповненого реєстру, з чим звіряти.
- `cancels_document_ref` (рівень 1 статусу чинності) і звірка рівня 2 (`probably_superseded`) — потребують реальної таблиці `facts` для порівняння.
- Морфологічна нормалізація ПІБ у називний відмінок (`normalize_nominative_case` — заглушка).
- OCR-free vision-екстракція напряму із зображення — залишено на майбутнє, не для пілота.

**Специфічно для `deployment_certificate`:** поле `stops` (кілька зупинок на зворотному боці бланка) відкладено — `priority: deferred` у схемі, тому автоматично потрапляє в `not_implemented_fields`, не вимагаючи окремого коду в цьому ноутбуці.